# §9C.2 — SNPE-C Posterior Calibration (10⁴ sims, CPU-only) — **v4**

Runs `pvgap_experiment/src/sbi_posterior.py` on the `lg_m50_healthy` case with `n_sim=10000`.

Colab runtime: **CPU High-RAM** (no GPU). Expect ~22–28 min.

**What's new in v4 (vs v3 which FAILed with mis-coverage 0.44)**
- Simulator now adds **relative Gaussian noise σ=2%** on the summary stats — otherwise the deterministic map makes SNPE-C learn a near-δ posterior.
- Hold-out θ\* is now sampled **from the prior itself** (guaranteed in-support) instead of a hard-coded reference box that could fall outside the emission support.
- Upload `pvgap_9C2_v3.zip` (bundled alongside this notebook).

**Outputs**: `results/sbi_prior_emit/9C2_calibration_N10k_v4.log` and `density_estimator_last.pt`.

**Gate**: per-dim mis-coverage at α=0.10 averaged over 3 hold-outs; |deviation from 0.10| ≤ 0.15 → §9C.2 PASS.

## 0. Runtime sanity

In [ ]:
import sys, platform, os
print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('cpu_count:', os.cpu_count())
!free -h | head -2

## 1. Install dependencies

In [ ]:
%pip install -q 'pybamm==25.12.2' 'sbi==0.26.1' scipy numpy
%pip install -q git+https://github.com/pybamm-team/pybamm-eis.git
import pybamm, sbi, pybammeis, torch
print('pybamm', pybamm.__version__, '| sbi', sbi.__version__, '| torch', torch.__version__)

## 2. Upload repo zip

Upload `pvgap_9C2_v3.zip` (contains v4-patched `sbi_posterior.py` + LRU-capped `pybamm_eis_residual.py`).

In [ ]:
from google.colab import files
up = files.upload()
zip_name = next(iter(up))
!rm -rf /content/pvgap_experiment && mkdir -p /content/pvgap_experiment && unzip -oq {zip_name} -d /content/pvgap_experiment
!ls /content/pvgap_experiment
!ls /content/pvgap_experiment/src | head
!ls /content/pvgap_experiment/results/sbi_prior_emit

## 2b. Sanity-check v4 patches are in place (fail-fast if old zip)

In [ ]:
src = open('/content/pvgap_experiment/src/sbi_posterior.py').read()
assert '_raw_flow' in src, 'old sbi_posterior.py — please upload pvgap_9C2_v3.zip'
assert 'density_estimator_last.pt' in src, 'old sbi_posterior.py (no ckpt save)'
assert '_SIGMA_REL' in src, 'v3 zip detected — please upload v4 bundle (pvgap_9C2_v3.zip)'
assert 'sigma_rel' in src, 'v3 zip detected'
print('OK — v4 sim-noise + prior hold-out patches present')
prr = open('/content/pvgap_experiment/src/pybamm_eis_residual.py').read()
assert '_SIM_CACHE_MAX' in prr, 'old pybamm_eis_residual.py (no LRU cap)'
print('OK — LRU cache cap present')

## 3. Smoke test: cold PyBaMM-EIS forward (~60–120 s first call)

In [ ]:
%cd /content/pvgap_experiment
import sys; sys.path.insert(0, '.')
import numpy as np, time
from src.pybamm_eis_residual import simulate_Z
f = np.logspace(-1, 3, 8)
t0 = time.time()
z = simulate_Z({'model_name':'SPM','parameter_set':'Chen2020','initial_soc':0.5,'frequencies':f})
print(f'cold sim: {time.time()-t0:.1f}s, |Z|={np.abs(z)}')
t0 = time.time()
_ = simulate_Z({'model_name':'SPM','parameter_set':'Chen2020','initial_soc':0.5,'frequencies':f})
print(f'warm (cached) sim: {time.time()-t0:.2f}s')

## 4. Run the 10⁴-sim calibration (v4)

Long step. Streams to log; the run survives a tab close.

In [ ]:
%cd /content/pvgap_experiment
!mkdir -p results/sbi_prior_emit
!pkill -9 -f sbi_posterior 2>/dev/null; true
!nohup python -u -m src.sbi_posterior --case_name lg_m50_healthy --n_sim 10000 --seed 0 \
  > results/sbi_prior_emit/9C2_calibration_N10k_v4.log 2>&1 &
import time; time.sleep(3)
!ps aux | grep sbi_posterior | grep -v grep
print('\nstreaming log (ctrl-M to stop; the run continues in background):')
!tail -f -n 200 results/sbi_prior_emit/9C2_calibration_N10k_v4.log

### Re-check progress without interrupting

In [ ]:
!ps -ef | grep sbi_posterior | grep -v grep
!tail -40 /content/pvgap_experiment/results/sbi_prior_emit/9C2_calibration_N10k_v4.log

## 5. Inspect result + download

Wait until the log shows `§9C.2 gate: ... PASS/FAIL`.

In [ ]:
!tail -60 /content/pvgap_experiment/results/sbi_prior_emit/9C2_calibration_N10k_v4.log
!ls -lh /content/pvgap_experiment/results/sbi_prior_emit/
from google.colab import files
files.download('/content/pvgap_experiment/results/sbi_prior_emit/9C2_calibration_N10k_v4.log')
files.download('/content/pvgap_experiment/results/sbi_prior_emit/density_estimator_last.pt')

## 6. (Optional) Persist to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp results/sbi_prior_emit/9C2_calibration_N10k_v4.log /content/drive/MyDrive/
# !cp results/sbi_prior_emit/density_estimator_last.pt /content/drive/MyDrive/